# Detecció d'opinions (pràctica 2.a)
### Recursos

- Movie Reviews Corpus

### Enunciat

- Implementeu un detector d'opinions positives o negatives amb alguns algoritmes d'aprenentatge supervisat de l'sklearn # perceptro, svm i arbres (3 models diferents)

- Utilitzeu com a dades el Movie Reviews Corpus de l'NLTK

- Dissenyeu i apliqueu un protocol de validació

- Utilitzeu el preprocés que cregueu més convenient: eliminació d'stop words, signes de puntuació... (reduir la dimensionalitat del vocabulari (amb lemes de diccionary o symsets))

- Utilitzeu el CountVectorizer per representar la informació

- Doneu la precisió (accuracy) i les matrius de confusió

- Analitzeu els resultats (Analitzar les frases que fallen)

num_features = 5000 <-- fixar mida de diccionari

## NLTK’s Movie Reviews Corpus

### Polarity corpus:

- 1000 exemples positius i 1000 negatius

### Requeriments:

In [8]:
import re
import nltk
nltk.download('movie_reviews')
from nltk.corpus import movie_reviews as mr

[nltk_data] Downloading package movie_reviews to
[nltk_data]     C:\Users\david\AppData\Roaming\nltk_data...
[nltk_data]   Package movie_reviews is already up-to-date!


### Ús:

In [3]:
mr.fileids('pos')[:2]

['pos/cv000_29590.txt', 'pos/cv001_18431.txt']

In [4]:
len(mr.fileids('neg'))

1000

In [5]:
mr.words()

['plot', ':', 'two', 'teen', 'couples', 'go', 'to', ...]

In [6]:
mr.raw(mr.fileids('pos')[:2][0]).split("\n")[:10]

["films adapted from comic books have had plenty of success , whether they're about superheroes ( batman , superman , spawn ) , or geared toward kids ( casper ) or the arthouse crowd ( ghost world ) , but there's never really been a comic book like from hell before . ",
 "for starters , it was created by alan moore ( and eddie campbell ) , who brought the medium to a whole new level in the mid '80s with a 12-part series called the watchmen . ",
 'to say moore and campbell thoroughly researched the subject of jack the ripper would be like saying michael jackson is starting to look a little odd . ',
 'the book ( or " graphic novel , " if you will ) is over 500 pages long and includes nearly 30 more that consist of nothing but footnotes . ',
 "in other words , don't dismiss this film because of its source . ",
 "if you can get past the whole comic book thing , you might find another stumbling block in from hell's directors , albert and allen hughes . ",
 "getting the hughes brothers to di

## CountVectorizer de l'sklearn

Codificador bag of words (TF-IDF)

### Exemple:

This is the first document.
This document is the second document.
And this is the third one.
Is this the first document?

### Matriu resultant:

0 1 1 1 0 0 1 0 1
0 2 0 1 0 1 1 0 1
1 0 0 1 1 0 1 1 1
0 1 1 1 0 0 1 0 1

### Diccionari:

index	word  
0:      and  
1:    document  
2:     first  
3:      is  
4:     one  
5:    second  
6:     the  
7:    third  
8:    this

### Referència:

https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html

In [7]:
import nltk
from nltk.corpus import movie_reviews
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import Perceptron
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report, accuracy_score
import random

# Assegureu-vos que teniu el conjunt de dades de Movie Reviews de NLTK
nltk.download('movie_reviews')

# Carregar les ressenyes de pel·lícules (les etiquetes són 'pos' i 'neg')
documents = [(movie_reviews.raw(fileid), category)
             for category in movie_reviews.categories()
             for fileid in movie_reviews.fileids(category)]

# Embarrejar les dades
random.shuffle(documents)

# Dividir les dades en textos (X) i etiquetes (y)
X = [doc[0] for doc in documents]
y = [doc[1] for doc in documents]

# Dividir en entrenament i test (80% entrenament, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Inicialitzem el CountVectorizer per convertir el text en una matriu de característiques
vectorizer = CountVectorizer(stop_words='english', max_features=5000)

# Model 1: Perceptró
perceptron = make_pipeline(vectorizer, Perceptron())
perceptron.fit(X_train, y_train)

# Model 2: SVM (Support Vector Machine)
svm = make_pipeline(vectorizer, SVC(kernel='linear'))
svm.fit(X_train, y_train)

# Model 3: Arbre de decisió
decision_tree = make_pipeline(vectorizer, DecisionTreeClassifier())
decision_tree.fit(X_train, y_train)

# Predicció en el conjunt de test per avaluar els models
y_pred_perceptron = perceptron.predict(X_test)
y_pred_svm = svm.predict(X_test)
y_pred_tree = decision_tree.predict(X_test)

# Resultats per a cada model
print("Perceptron Performance:")
print(classification_report(y_test, y_pred_perceptron))

print("SVM Performance:")
print(classification_report(y_test, y_pred_svm))

print("Decision Tree Performance:")
print(classification_report(y_test, y_pred_tree))

# Avaluació de la precisió
print("Accuracy for Perceptron:", accuracy_score(y_test, y_pred_perceptron))
print("Accuracy for SVM:", accuracy_score(y_test, y_pred_svm))
print("Accuracy for Decision Tree:", accuracy_score(y_test, y_pred_tree))

[nltk_data] Downloading package movie_reviews to
[nltk_data]     C:\Users\david\AppData\Roaming\nltk_data...
[nltk_data]   Package movie_reviews is already up-to-date!


Perceptron Performance:
              precision    recall  f1-score   support

         neg       0.87      0.81      0.84       215
         pos       0.79      0.85      0.82       185

    accuracy                           0.83       400
   macro avg       0.83      0.83      0.83       400
weighted avg       0.83      0.83      0.83       400

SVM Performance:
              precision    recall  f1-score   support

         neg       0.88      0.85      0.86       215
         pos       0.83      0.86      0.85       185

    accuracy                           0.85       400
   macro avg       0.85      0.86      0.85       400
weighted avg       0.86      0.85      0.86       400

Decision Tree Performance:
              precision    recall  f1-score   support

         neg       0.68      0.66      0.67       215
         pos       0.62      0.65      0.63       185

    accuracy                           0.65       400
   macro avg       0.65      0.65      0.65       400
weight

In [6]:
vec = CountVectorizer(stop_words='english', max_features=5000)
vec.fit_transform(X_train)
print(len(vec.get_feature_names_out()))

5000


# Detecció d'opinions (pràctica 2.b)

### Enunciat

- Implementeu un detector d'opinions positives o negatives no supervisat

    1. Apliqueu l'UKB per obtenir els synsets de les paraules (nltk lesk)
    2. Obtingueu els valors SentiWordnet de cada synset

- Utilitzeu com a dades el/els conjunts de test que hagueu utilitzat a la pràctica 2.a

- Penseu en com podeu combinar aquests valors per obtenir un resultat

- Penseu que fareu si el synset no hi és a SentiWordnet

- Penseu quines categories utilitzareu:

    - només adjectius
    - noms, adjectius i adverbis
    - noms, adjectius, verbs i adverbis (com a minim fer aquestes combinacions)

- Analitzeu els resultats i compareu-los amb els de la part supervisada

    - Desambiguació de 10 frases amb lesk and UKB i comparar la desambiguació que han fet -----> (Com que la diferencia no es tan gran farem servir lesk)

svm, perceptro, preprocessat | noms, adjectius i adverbis i pensar només adjectius

In [9]:
# UKB
from textserver import TextServer
# Lesk
from nltk.corpus import wordnet as wn
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger')
wnl = nltk.stem.WordNetLemmatizer()
def lemmatize(p):
  d = {'NN': 'n', 'NNS': 'n', 
       'JJ': 'a', 'JJR': 'a', 'JJS': 'a', 
       'VB': 'v', 'VBD': 'v', 'VBG': 'v', 'VBN': 'v', 'VBP': 'v', 'VBZ': 'v', 
       'RB': 'r', 'RBR': 'r', 'RBS': 'r'}
  if p[1] in d:
    return wnl.lemmatize(p[0], pos=d[p[1]]), d[p[1]]
  return p[0], None

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\david\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\david\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\david\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


In [10]:
frase = mr.raw(mr.fileids('pos')[:2][0]).split("\n")[0] # Lesk
words = frase.split()
tags = nltk.pos_tag(words)
resultat_lesk = []
for pair in tags:
    lem = lemmatize(pair)
    if lem[1] != None:
        synset = nltk.wsd.lesk(frase, *lem)
        if synset:
            resultat_lesk.append((pair[0],synset.name(), synset.definition()))
        else:
            resultat_lesk.append((pair[0],"", ""))

In [11]:
frase = mr.raw(mr.fileids('pos')[:2][0]).split("\n")[0] # UKB
ts = TextServer('davidvazquez', 'c4tSX16B!', 'senses') 
synset_table = ts.senses(frase)
resultat_ukb = []
for frase_synset in synset_table:
    for synset in frase_synset:
        if synset[4] != 'N/A':
            syn = wn.synset_from_pos_and_offset(synset[4][-1], int(synset[4][:-2]))
            resultat_ukb.append((synset[0],syn.name(), syn.definition()))
        else:
            resultat_lesk.append((synset[0],"", ""))
    print()

HTTPError: 500 Server Error:  for url: http://frodo.lsi.upc.edu:8080/TextWS/textservlet/ws/processQuery/senses

In [ ]:
print(frase)

i = 0
j = 0
while i < len(resultat_lesk):
    # Print lesk synset
    print(f"lesk: {resultat_lesk[i][0]} {resultat_lesk[i][1]} {resultat_lesk[i][2]}", end=" ")

    k = j
    while k < len(resultat_ukb):
        # Check if words match
        if resultat_lesk[i][0] == resultat_ukb[k][0]:
            # Print matched synsets side by side
            print(f"{' ' * (150 - len(resultat_lesk[i][0]) - len(resultat_lesk[i][1]) - len(resultat_lesk[i][2]))} ukb: {resultat_ukb[k][0]} {resultat_ukb[k][1]} {resultat_ukb[k][2]}")
            break
        k += 1
    
    # Move the index for lesk and ukb to next unmatched synset
    j = k
    i += 1

# Print remaining ukb synsets if they don’t match any from lesk
while j < len(resultat_ukb):
    print(f"{' ' * 150} ukb: {resultat_ukb[j][0]} {resultat_ukb[j][1]} {resultat_ukb[j][2]}")
    j += 1

films adapted from comic books have had plenty of success , whether they're about superheroes ( batman , superman , spawn ) , or geared toward kids ( casper ) or the arthouse crowd ( ghost world ) , but there's never really been a comic book like from hell before . 
films movie.n.01 a form of entertainment that enacts a story by sound and a sequence of images giving the illusion of continuous movement
adapted adapt.v.01 make fit for, or change to suit a new purpose
                                                                                         films movie.n.01 a form of entertainment that enacts a story by sound and a sequence of images giving the illusion of continuous movement
comic comic.a.02 of or relating to or characteristic of comedy
                                                                                           adapted adapt.v.01 make fit for, or change to suit a new purpose
books script.n.01 a written version of a play or other dramatic composition; used in

In [11]:
deu_frases = mr.raw(mr.fileids('pos')[:2][0]).split("\n")[:10]
for frase in deu_frases:
    tags = nltk.pos_tag(frase)
    synset = nltk.wsd.lesk(frase, 'bank', 'n')
    synset.name(), synset.definition()

TypeError: tokens: expected a list of strings, got a string

In [12]:
import nltk
from nltk.corpus import movie_reviews
from nltk.corpus import sentiwordnet as swn
from nltk.wsd import lesk
from nltk.tokenize import word_tokenize

# Baixar recursos
nltk.download('movie_reviews')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('sentiwordnet')

[nltk_data] Downloading package movie_reviews to
[nltk_data]     C:\Users\david\AppData\Roaming\nltk_data...
[nltk_data]   Package movie_reviews is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\david\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\david\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package sentiwordnet to
[nltk_data]     C:\Users\david\AppData\Roaming\nltk_data...
[nltk_data]   Package sentiwordnet is already up-to-date!


True

In [14]:
# Cargar les ressenyes de movie_reviews
reviews = [(list(movie_reviews.words(fileid)), category)
           for fileid in movie_reviews.fileids()
           for category in [fileid.split('/')[0]]]

# Mostrem les primeres ressenyes
print(reviews[:2])  # Mostra les dues primeres ressenyes

[(['plot', ':', 'two', 'teen', 'couples', 'go', 'to', 'a', 'church', 'party', ',', 'drink', 'and', 'then', 'drive', '.', 'they', 'get', 'into', 'an', 'accident', '.', 'one', 'of', 'the', 'guys', 'dies', ',', 'but', 'his', 'girlfriend', 'continues', 'to', 'see', 'him', 'in', 'her', 'life', ',', 'and', 'has', 'nightmares', '.', 'what', "'", 's', 'the', 'deal', '?', 'watch', 'the', 'movie', 'and', '"', 'sorta', '"', 'find', 'out', '.', '.', '.', 'critique', ':', 'a', 'mind', '-', 'fuck', 'movie', 'for', 'the', 'teen', 'generation', 'that', 'touches', 'on', 'a', 'very', 'cool', 'idea', ',', 'but', 'presents', 'it', 'in', 'a', 'very', 'bad', 'package', '.', 'which', 'is', 'what', 'makes', 'this', 'review', 'an', 'even', 'harder', 'one', 'to', 'write', ',', 'since', 'i', 'generally', 'applaud', 'films', 'which', 'attempt', 'to', 'break', 'the', 'mold', ',', 'mess', 'with', 'your', 'head', 'and', 'such', '(', 'lost', 'highway', '&', 'memento', ')', ',', 'but', 'there', 'are', 'good', 'and', '

In [15]:
def get_lesk_synset(word, context):
    # Intentem obtenir el synset de la paraula utilitzant 'lesk'
    return lesk(context, word)

# Exemple d'ús:
context = word_tokenize("This movie was incredibly good and entertaining")
word = "good"
synset = get_lesk_synset(word, context)

if synset:
    print(f"Synset de {word}: {synset}")
else:
    print(f"No s'ha trobat synset per la paraula {word}")

Synset de good: Synset('well.r.01')


In [16]:
def get_sentiment_from_synset(synset):
    if synset:
        senti_synset = swn.senti_synset(synset.name())
        return senti_synset.pos_score() - senti_synset.neg_score()
    return 0.0  # Si no trobem cap synset, retornem un sentiment neutral

# Exemple de càlcul del sentiment
sentiment = get_sentiment_from_synset(synset)
print(f"Sentiment: {sentiment}")

Sentiment: 0.375


In [55]:
def analyze_review_sentiment(review, accepted = ["n", "v", "r", "a"]):
    sentiment_score = 0.0
    
    # Tokenitzem la ressenya i analitzem cada paraula
    tags = nltk.pos_tag(review)
    for pair in tags:
        lem = lemmatize(pair)
        if lem[1] != None and lem[1] in accepted:
            synset = nltk.wsd.lesk(review, *lem)
            sentiment_score += get_sentiment_from_synset(synset)

    # Determinem el sentiment global de la ressenya
    if sentiment_score > 0:
        return "pos"
    elif sentiment_score < 0:
        return "neg"
    else:
        return "neutral"

# Analitzem les primeres 10 ressenyes
for review, category in reviews[:10]:
    predicted_sentiment = analyze_review_sentiment(review)
    print(f"Ressenya: {' '.join(review)}")
    print(f"Sentiment real: {category}, Sentiment predir: {predicted_sentiment}")
    print("="*50)

Ressenya: plot : two teen couples go to a church party , drink and then drive . they get into an accident . one of the guys dies , but his girlfriend continues to see him in her life , and has nightmares . what ' s the deal ? watch the movie and " sorta " find out . . . critique : a mind - fuck movie for the teen generation that touches on a very cool idea , but presents it in a very bad package . which is what makes this review an even harder one to write , since i generally applaud films which attempt to break the mold , mess with your head and such ( lost highway & memento ) , but there are good and bad ways of making all types of films , and these folks just didn ' t snag this one correctly . they seem to have taken this pretty neat concept , but executed it terribly . so what are the problems with the movie ? well , its main problem is that it ' s simply too jumbled . it starts off " normal " but then downshifts into this " fantasy " world in which you , as an audience member , ha

In [ ]:
# Extraem les etiquetes reals i les prediccions
real_labels = [category for _, category in reviews]
predicted_labels = [analyze_review_sentiment(review, ["a"]) for review, _ in reviews]

# Calcular l'accuracy
accuracy = accuracy_score(real_labels, predicted_labels)
print(f"Accuracy: {accuracy * 100:.2f}%")

Accuracy: 59.70%


In [ ]:
# Extraem les etiquetes reals i les prediccions
real_labels = [category for _, category in reviews]
predicted_labels = [analyze_review_sentiment(review, accepted = ["a", "r"]) for review, _ in reviews]

# Calcular l'accuracy
accuracy = accuracy_score(real_labels, predicted_labels)
print(f"Accuracy: {accuracy * 100:.2f}%")

Accuracy: 60.50%


In [59]:
# Extraem les etiquetes reals i les prediccions
real_labels = [category for _, category in reviews]
predicted_labels = [analyze_review_sentiment(review, accepted = ["a", "r", "v"]) for review, _ in reviews]

# Calcular l'accuracy
accuracy = accuracy_score(real_labels, predicted_labels)
print(f"Accuracy: {accuracy * 100:.2f}%") 

Accuracy: 56.40%


In [60]:
# Extraem les etiquetes reals i les prediccions
real_labels = [category for _, category in reviews]
predicted_labels = [analyze_review_sentiment(review, accepted = ["a", "r", "v", "n"]) for review, _ in reviews]

# Calcular l'accuracy
accuracy = accuracy_score(real_labels, predicted_labels)
print(f"Accuracy: {accuracy * 100:.2f}%") 

Accuracy: 55.75%
